# Gemini Vision API Explorer - Structured Output
This notebook demonstrates how to send an image to the Gemini API and receive a **structured JSON response**.

In [ ]:
import os
import google.generativeai as genai
from PIL import Image
from dotenv import load_dotenv
from IPython.display import display
import typing_extensions as typing
import json

# Load environment variables
load_dotenv(dotenv_path="../.env")

# Configure Gemini API
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment variables.")

genai.configure(api_key=api_key)

## Define the Output Structure
We use `typing.TypedDict` to define the schema that Gemini should follow.

In [ ]:
class ImageAnalysis(typing.TypedDict):
    description: str
    objects: list[str]
    confidence_score: float
    interesting_facts: list[str]
    dominant_colors: list[str]

## List available images

In [ ]:
image_folder = "../images"
images = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(images)} images: {images}")

## Select and Visualize an Image

In [ ]:
# Change index to try different images
selected_image = images[0] 
img_path = os.path.join(image_folder, selected_image)

img = Image.open(img_path)
display(img)
print(f"Selected: {selected_image}")

## Call Gemini API with Structured Output

In [ ]:
model = genai.GenerativeModel('gemini-1.5-flash')

prompt = "Analyze this image and provide a detailed description, list visible objects, give a confidence level for your analysis, mention 2-3 interesting facts related to the content, and identify the dominant colors."

print("Analyzing...")
response = model.generate_content(
    [prompt, img],
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        response_schema=ImageAnalysis
    )
)

# Parse and display results
analysis = json.loads(response.text)
print(json.dumps(analysis, indent=2))